In [1]:
from pathlib import Path
import torch
import torch.nn as nn
from config import get_config, latest_weights_file_path
from train import get_model, get_ds, run_validation
from accuracy import translate_sentence
from translate import translate

import numpy as np

c:\Users\nikol\Desktop\UNIBO\Terzo Anno\Tesi\Transformer\Transformer2.0\envUmar\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\nikol\Desktop\UNIBO\Terzo Anno\Tesi\Transformer\Transformer2.0\envUmar\lib\site-packages\torchaudio\backend\utils.py:74: UserWarning: No audio backend is available.
  warnings.warn("No audio backend is available.")


In [2]:
# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
config = get_config()
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_ds(config)
model = get_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)

# Load the pretrained weights
model_filename = latest_weights_file_path(config)
state = torch.load(model_filename, map_location=device)
model.load_state_dict(state['model_state_dict'])

Using device: cpu
Max length of source sentence: 309
Max length of target sentence: 274


<All keys matched successfully>

In [3]:
run_validation(model, val_dataloader, tokenizer_src, tokenizer_tgt, config['seq_len'], device, lambda msg: print(msg), 0, None, num_examples=10)

--------------------------------------------------------------------------------
    SOURCE: About two o'clock p.m. I entered the village.
    TARGET: Giunsi al villaggio verso le due.
 PREDICTED: Giunsi al villaggio verso le due .
--------------------------------------------------------------------------------
    SOURCE: No wonder you have rather the look of another world. I marvelled where you had got that sort of face.
    TARGET: Non mi sorprende più che abbiate una faccia come una morta.
 PREDICTED: Non mi sorprende se l ' abbiate una faccia come una morta .
--------------------------------------------------------------------------------
    SOURCE: "Gentlemen, you hear!
    TARGET: — Signori, l'avete udita?
 PREDICTED: — Signori , l ' avete udita ?
--------------------------------------------------------------------------------
    SOURCE: It brings him also no profit, but pure loss.'
    TARGET: Anche senza tornaconto. Proprio in perdita.
 PREDICTED: Anche lui non può essere ut

In [14]:
s = "And I came out immediately, for I trembled at the idea of being dragged forth by the said Jack."
t = translate_sentence(s, model, tokenizer_src, tokenizer_tgt, config['seq_len'], device)
print("SOURCE: ", s)
print("PREDICTED:", t)

SOURCE:  And I came out immediately, for I trembled at the idea of being dragged forth by the said Jack.
PREDICTED: Uscii subito , perché mi al pensiero di esser condotta fuori dal mio nascondiglio da John .


In [ ]:
# Metrics on validation set

import torchmetrics
from train import greedy_decode

def evaluate_metrics(model, val_dataloader, tokenizer_src, tokenizer_tgt, 
                     seq_len, device, num_examples=50):
    model.eval()
    
    source_texts = []
    expected = []
    predicted = []

    with torch.no_grad():
        for i, batch in enumerate(val_dataloader):
            if i >= num_examples:
                break

            encoder_input = batch["encoder_input"].to(device)
            encoder_mask  = batch["encoder_mask"].to(device)

            assert encoder_input.size(0) == 1, "Batch size must be 1"

            model_out = greedy_decode(
                model, encoder_input, encoder_mask,
                tokenizer_src, tokenizer_tgt, seq_len, device
            )

            source_texts.append(batch["src_text"][0])
            expected.append(batch["tgt_text"][0])
            predicted.append(tokenizer_tgt.decode(model_out.detach().cpu().numpy()))

    # --- Metriche ---
    cer_metric  = torchmetrics.CharErrorRate()
    wer_metric  = torchmetrics.WordErrorRate()
    bleu_metric = torchmetrics.BLEUScore()

    expected_bleu = [[e] for e in expected]

    cer  = cer_metric(predicted, expected).item()
    wer  = wer_metric(predicted, expected).item()
    bleu = bleu_metric(predicted, expected_bleu).item()

    # --- Stampa risultati ---
    sep = "=" * 50
    print(sep)
    print(f"  Valutazione su {num_examples} esempi")
    print(sep)
    print(f"  BLEU score      : {bleu:.4f}  (↑ migliore)")
    print(f"  Word Error Rate : {wer:.4f}  (↓ migliore)")
    print(f"  Char Error Rate : {cer:.4f}  (↓ migliore)")
    print(sep)

    # --- Qualche esempio per sanity check ---
    print("\nCampione predizioni:\n")
    for src, tgt, pred in zip(source_texts[:3], expected[:3], predicted[:3]):
        print(f"  SRC  : {src}")
        print(f"  TGT  : {tgt}")
        print(f"  PRED : {pred}")
        print()

    return {"bleu": bleu, "wer": wer, "cer": cer}

# --- Esegui ---
metrics = evaluate_metrics(
    model, val_dataloader, tokenizer_src, tokenizer_tgt,
    config['seq_len'], device, num_examples=50
)

c:\Users\nikol\Desktop\UNIBO\Terzo Anno\Tesi\Transformer\Transformer2.0\envUmar\lib\site-packages\torchmetrics\utilities\prints.py:61: FutureWarning: Importing `CharErrorRate` from `torchmetrics` was deprecated and will be removed in 2.0. Import `CharErrorRate` from `torchmetrics.text` instead.
  _future_warning(
c:\Users\nikol\Desktop\UNIBO\Terzo Anno\Tesi\Transformer\Transformer2.0\envUmar\lib\site-packages\torchmetrics\utilities\prints.py:61: FutureWarning: Importing `WordErrorRate` from `torchmetrics` was deprecated and will be removed in 2.0. Import `WordErrorRate` from `torchmetrics.text` instead.
  _future_warning(
c:\Users\nikol\Desktop\UNIBO\Terzo Anno\Tesi\Transformer\Transformer2.0\envUmar\lib\site-packages\torchmetrics\utilities\prints.py:61: FutureWarning: Importing `BLEUScore` from `torchmetrics` was deprecated and will be removed in 2.0. Import `BLEUScore` from `torchmetrics.text` instead.
  _future_warning(


  Valutazione su 50 esempi
  BLEU score      : 0.4288  (↑ migliore)
  Word Error Rate : 0.4967  (↓ migliore)
  Char Error Rate : 0.1521  (↓ migliore)

Campione predizioni:

  SRC  : 'The first thing I've got to do,' said Alice to herself, as she wandered about in the wood, 'is to grow to my right size again; and the second thing is to find my way into that lovely garden.
  TGT  : “La prima cosa che dovrò fare, — pensò Alice, vagando nel bosco, — è di ricrescere e giungere alla mia statura normale; la seconda, di trovare la via per entrare in quel bel giardino.
  PRED : “ La prima cosa che dovrò fare , — pensò Alice , vagando nel bosco , — è di ricrescere e giungere alla mia statura normale ; la seconda , di trovare la via per entrare in quel bel giardino .

  SRC  : We put the kettle on to boil, up in the nose of the boat, and went down to the stern and pretended to take no notice of it, but set to work to get the other things out.
  TGT  : A prua mettemmo a bollire il calderino del tè

In [39]:
# --- Valutazione su testSet.txt con metriche WER, CER, BLEU ---
from torch.utils.data import DataLoader, Dataset

class TestSetDataset(Dataset):
    def __init__(self, text_file, tokenizer_src, seq_len):
        self.seq_len = seq_len
        self.tokenizer_src = tokenizer_src
        
        # Leggi le frasi dal file
        with open(text_file, 'r', encoding='utf-8') as f:
            self.sentences = [line.strip() for line in f if line.strip()]
    
    def __len__(self):
        return len(self.sentences)
    
    def __getitem__(self, idx):
        src_text = self.sentences[idx]
        
        # Encoding
        src_tokens = self.tokenizer_src.encode(src_text).ids
        src_tokens = src_tokens[:self.seq_len - 2]
        src_tokens = [self.tokenizer_src.token_to_id('[SOS]')] + src_tokens + [self.tokenizer_src.token_to_id('[EOS]')]
        
        # Padding
        src_tokens += [self.tokenizer_src.token_to_id('[PAD]')] * (self.seq_len - len(src_tokens))
        src_mask = (torch.tensor(src_tokens) != self.tokenizer_src.token_to_id('[PAD]')).unsqueeze(0).unsqueeze(0).int()
        
        return {
            'encoder_input': torch.tensor(src_tokens, dtype=torch.long),
            'encoder_mask': src_mask,
            'src_text': src_text,
        }

# Crea il dataset e il dataloader
testset_dataset = TestSetDataset('testSet.txt', tokenizer_src, config['seq_len'])
testset_loader = DataLoader(testset_dataset, batch_size=1, shuffle=False)

# Genera le predizioni
print("\n" + "="*70)
print("VALUTAZIONE SUL FILE testSet.txt (100 frasi)")
print("="*70 + "\n")

model.eval()
source_texts = []
predictions = []

with torch.no_grad():
    for i, batch in enumerate(testset_loader):
        encoder_input = batch["encoder_input"].to(device)
        encoder_mask = batch["encoder_mask"].to(device)
        src_text = batch["src_text"][0]
        
        model_out = greedy_decode(
            model, encoder_input, encoder_mask,
            tokenizer_src, tokenizer_tgt, config['seq_len'], device
        )
        
        pred_text = tokenizer_tgt.decode(model_out.detach().cpu().numpy())
        source_texts.append(src_text)
        predictions.append(pred_text)
        
        if i < 5:  # Mostra i primi 5 esempi
            print(f"Esempio {i+1}:")
            print(f"  SRC  : {src_text}")
            print(f"  PRED : {pred_text}")
            print()

print(f"✓ Valutazione completata su {len(predictions)} frasi")
print(f"✓ Traduzioni generate: {len([p for p in predictions if p])}")

# Salva le predizioni
with open('testSet_predictions.txt', 'w', encoding='utf-8') as f:
    for pred in predictions:
        f.write(pred + '\n')
print(f"✓ Predizioni salvate in 'testSet_predictions.txt'\n")

# ============== CALCOLO METRICHE ==============
# NOTA: Per calcolare WER, CER e BLEU servono i testi di riferimento (ground truth in italiano)
# Se hai un file testSetTarget.txt con le traduzioni italiane di riferimento, 
# puoi usare il codice sottostante

try:
    # Prova a leggere le traduzioni di riferimento
    with open('testSetTarget.txt', 'r', encoding='utf-8') as f:
        expected = [line.strip() for line in f if line.strip()]
    
    if len(expected) == len(predictions):
        print("="*70)
        print("CALCOLO DELLE METRICHE")
        print("="*70)
        
        # Calcola le metriche
        cer_metric  = torchmetrics.CharErrorRate()
        wer_metric  = torchmetrics.WordErrorRate()
        bleu_metric = torchmetrics.BLEUScore()
        
        expected_bleu = [[e] for e in expected]
        
        cer  = cer_metric(predictions, expected).item()
        wer  = wer_metric(predictions, expected).item()
        bleu = bleu_metric(predictions, expected_bleu).item()
        
        # Stampa risultati
        print(f"\nRisultati su {len(predictions)} frasi:")
        print(f"  BLEU score      : {bleu:.4f}  (↑ migliore)")
        print(f"  Word Error Rate : {wer:.4f}  (↓ migliore)")
        print(f"  Char Error Rate : {cer:.4f}  (↓ migliore)\n")
    else:
        print(f"⚠ Il file testSetTarget.txt ha {len(expected)} righe ma servono {len(predictions)}")
        
except FileNotFoundError:
    print("⚠ File 'testSetTarget.txt' non trovato")
    print("  Per calcolare WER, CER e BLEU serve un file italiano di riferimento")
    print("  Crea testSetTarget.txt con le traduzioni di riferimento (una per riga)")


VALUTAZIONE SUL FILE testSet.txt (100 frasi)

Esempio 1:
  SRC  : The quick brown fox jumps over the lazy dog.
  PRED : Un piccolo prete andato coi gomiti e il cane .

Esempio 2:
  SRC  : Good morning, how are you feeling today?
  PRED : Buona mattina , come avete fatto il giorno ?

Esempio 3:
  SRC  : I would like to order a coffee with milk and sugar.
  PRED : a fare il caffè e il caffè .

Esempio 4:
  SRC  : The weather is beautiful this time of year in Italy.
  PRED : Il tempo è bello benissimo .

Esempio 5:
  SRC  : Can you help me find my way to the train station?
  PRED : Potete dirmi del treno con la stazione del treno ?

✓ Valutazione completata su 100 frasi
✓ Traduzioni generate: 100
✓ Predizioni salvate in 'testSet_predictions.txt'

CALCOLO DELLE METRICHE

Risultati su 100 frasi:
  BLEU score      : 0.0306  (↑ migliore)
  Word Error Rate : 1.0640  (↓ migliore)
  Char Error Rate : 0.6864  (↓ migliore)



In [ ]:
# --- Carica il file ita.txt di Kaggle (formato TSV: EN \t IT \t metadata) ---
import pandas as pd

print("Caricamento del file ita.txt...\n")

try:
    # Leggi il file TSV
    # Colonne: English | Italian | Metadata
    df = pd.read_csv('ita.txt', sep='\t', header=None, names=['English', 'Italian', 'Metadata'])
    
    print(f"✓ File caricato: {len(df)} righe")
    print(f"✓ Colonne: {df.columns.tolist()}")
    print(f"\nPrime 5 righe:")
    print(df.head())
    
    # Rimuovi eventuali righe vuote
    df = df.dropna(subset=['English', 'Italian'])
    print(f"\n✓ Dopo pulizia: {len(df)} coppie EN-IT valide")
    
except FileNotFoundError:
    print("⚠ File 'ita.txt' non trovato nella directory corrente")
    df = None
except Exception as e:
    print(f"⚠ Errore nel caricamento: {e}")
    df = None


Caricamento del file ita.txt...

✓ File caricato: 341554 righe
✓ Colonne: ['English', 'Italian', 'Metadata']

Prime 5 righe:
  English   Italian                                           Metadata
0     Hi.     Ciao!  CC-BY 2.0 (France) Attribution: tatoeba.org #5...
1    Run!    Corri!  CC-BY 2.0 (France) Attribution: tatoeba.org #9...
2    Run!    Corra!  CC-BY 2.0 (France) Attribution: tatoeba.org #9...
3    Run!  Correte!  CC-BY 2.0 (France) Attribution: tatoeba.org #9...
4    Who?      Chi?  CC-BY 2.0 (France) Attribution: tatoeba.org #2...

✓ Dopo pulizia: 341554 coppie EN-IT valide


In [ ]:
# --- Valutazione con metriche WER, CER, BLEU su 100 coppie CASUALI di Kaggle ---
from torch.utils.data import DataLoader, Dataset
import json

# Verifica che il dataframe sia stato caricato
if 'df' not in locals() or df is None:
    print("⚠ ERRORE: df non è definito!")
    print("Esegui prima la cella di caricamento del file ita.txt")
else:
    # Identifica le colonne
    en_col = 'English' if 'English' in df.columns else df.columns[0]
    it_col = 'Italian' if 'Italian' in df.columns else df.columns[1]

    print(f"Usando colonne: EN='{en_col}', IT='{it_col}'")
    print(f"Dataset totale: {df.shape[0]} righe\n")

    # ===== CAMPIONA 100 COPPIE CASUALI =====
    num_samples = min(100, len(df))
    df_sample = df.sample(n=num_samples, random_state=42)
    print(f"✓ Campione: {len(df_sample)} coppie casuali (random_state=42)")
    print(f"Prime 3 coppie del campione:")
    for idx, row in df_sample.head(3).iterrows():
        print(f"  EN: {row[en_col][:60]}...")
        print(f"  IT: {row[it_col][:60]}...\n")

    class LocalDataset(Dataset):
        def __init__(self, dataframe, en_col, it_col, tokenizer_src, seq_len):
            self.seq_len = seq_len
            self.tokenizer_src = tokenizer_src
            self.en_texts = dataframe[en_col].tolist()
            self.it_texts = dataframe[it_col].tolist()
        
        def __len__(self):
            return len(self.en_texts)
        
        def __getitem__(self, idx):
            src_text = str(self.en_texts[idx]).strip()
            tgt_text = str(self.it_texts[idx]).strip()
            
            # Encoding inglese
            src_tokens = self.tokenizer_src.encode(src_text).ids
            src_tokens = src_tokens[:self.seq_len - 2]
            src_tokens = [self.tokenizer_src.token_to_id('[SOS]')] + src_tokens + [self.tokenizer_src.token_to_id('[EOS]')]
            src_tokens += [self.tokenizer_src.token_to_id('[PAD]')] * (self.seq_len - len(src_tokens))
            src_mask = (torch.tensor(src_tokens) != self.tokenizer_src.token_to_id('[PAD]')).unsqueeze(0).unsqueeze(0).int()
            
            return {
                'encoder_input': torch.tensor(src_tokens, dtype=torch.long),
                'encoder_mask': src_mask,
                'src_text': src_text,
                'tgt_text': tgt_text,
            }

    # Crea il dataset con il campione
    local_dataset = LocalDataset(df_sample, en_col, it_col, tokenizer_src, config['seq_len'])
    local_loader = DataLoader(local_dataset, batch_size=1, shuffle=False)

    # Genera le predizioni
    print("\n" + "="*70)
    print(f"VALUTAZIONE ({len(local_dataset)} coppie EN-IT CASUALI)")
    print("="*70 + "\n")

    model.eval()
    source_texts = []
    expected = []
    predictions = []

    with torch.no_grad():
        for i, batch in enumerate(local_loader):
            encoder_input = batch["encoder_input"].to(device)
            encoder_mask = batch["encoder_mask"].to(device)
            src_text = batch["src_text"][0]
            tgt_text = batch["tgt_text"][0]
            
            model_out = greedy_decode(
                model, encoder_input, encoder_mask,
                tokenizer_src, tokenizer_tgt, config['seq_len'], device
            )
            
            pred_text = tokenizer_tgt.decode(model_out.detach().cpu().numpy())
            source_texts.append(src_text)
            expected.append(tgt_text)
            predictions.append(pred_text)
            
            if i < 5:  # Mostra i primi 5 esempi
                print(f"Esempio {i+1}:")
                print(f"  EN   : {src_text}")
                print(f"  IT (ref) : {tgt_text}")
                print(f"  IT (pred): {pred_text}")
                print()

    # CALCOLO METRICHE
    print("\n" + "="*70)
    print("CALCOLO DELLE METRICHE (WER, CER, BLEU)")
    print("="*70)

    cer_metric  = torchmetrics.CharErrorRate()
    wer_metric  = torchmetrics.WordErrorRate()
    bleu_metric = torchmetrics.BLEUScore()

    expected_bleu = [[e] for e in expected]

    cer  = cer_metric(predictions, expected).item()
    wer  = wer_metric(predictions, expected).item()
    bleu = bleu_metric(predictions, expected_bleu).item()

    # Stampa risultati
    print(f"\nRisultati su {len(predictions)} frasi casuali:")
    print(f"  BLEU score      : {bleu:.4f}  (↑ migliore)")
    print(f"  Word Error Rate : {wer:.4f}  (↓ migliore)")
    print(f"  Char Error Rate : {cer:.4f}  (↓ migliore)\n")

    # Salva i risultati
    summary = {
        'num_examples': len(predictions),
        'bleu': bleu,
        'wer': wer,
        'cer': cer,
        'random_state': 42,
        'examples': [
            {'en': src, 'it_ref': tgt, 'it_pred': pred}
            for src, tgt, pred in zip(source_texts[:10], expected[:10], predictions[:10])
        ]
    }

    with open('evaluation_results.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print(f"✓ Risultati salvati in 'evaluation_results.json'")


Usando colonne: EN='English', IT='Italian'
Dataset totale: 341554 righe

✓ Campione: 100 coppie casuali (random_state=42)
Prime 3 coppie del campione:
  EN: Maybe Tom can do that for us....
  IT: Forse Tom lo può fare per noi....

  EN: You're energetic....
  IT: Sei energica....

  EN: Do you like doing this?...
  IT: Le piace fare questo?...


VALUTAZIONE (100 coppie EN-IT CASUALI)

Esempio 1:
  EN   : Maybe Tom can do that for us.
  IT (ref) : Forse Tom lo può fare per noi.
  IT (pred): Questo va ad noi giudicare .

Esempio 2:
  EN   : You're energetic.
  IT (ref) : Sei energica.
  IT (pred): Voi in un momento all ’ orso .

Esempio 3:
  EN   : Do you like doing this?
  IT (ref) : Le piace fare questo?
  IT (pred): Ti così ?

Esempio 4:
  EN   : He hesitated for a moment.
  IT (ref) : Esitò per un momento.
  IT (pred): Egli rimase in un attimo pensosa , per la indecisione .

Esempio 5:
  EN   : I don't have any other questions.
  IT (ref) : Io non ho altre domande.
  IT (pred): Non h

c:\Users\nikol\Desktop\UNIBO\Terzo Anno\Tesi\Transformer\Transformer2.0\envUmar\lib\site-packages\torchmetrics\utilities\prints.py:61: FutureWarning: Importing `CharErrorRate` from `torchmetrics` was deprecated and will be removed in 2.0. Import `CharErrorRate` from `torchmetrics.text` instead.
  _future_warning(
c:\Users\nikol\Desktop\UNIBO\Terzo Anno\Tesi\Transformer\Transformer2.0\envUmar\lib\site-packages\torchmetrics\utilities\prints.py:61: FutureWarning: Importing `WordErrorRate` from `torchmetrics` was deprecated and will be removed in 2.0. Import `WordErrorRate` from `torchmetrics.text` instead.
  _future_warning(
c:\Users\nikol\Desktop\UNIBO\Terzo Anno\Tesi\Transformer\Transformer2.0\envUmar\lib\site-packages\torchmetrics\utilities\prints.py:61: FutureWarning: Importing `BLEUScore` from `torchmetrics` was deprecated and will be removed in 2.0. Import `BLEUScore` from `torchmetrics.text` instead.
  _future_warning(
